In [31]:
import subprocess
import cv2
import os
import numpy as np
import re
from scipy.optimize import linear_sum_assignment

# ===== Configuration =====
darknet_path = "/mnt/c/Users/satar/OneDrive/Desktop/TRACKTER/CODE/COMPUTER_VISION/KEY_FRAME_SELECTION/RUNNING/darknet/darknet/darknet"
cfg_path = "/mnt/c/Users/satar/OneDrive/Desktop/TRACKTER/CODE/COMPUTER_VISION/KEY_FRAME_SELECTION/RUNNING/darknet/darknet/cfg/yolov4-tiny-custom.cfg"
weights_path = "/mnt/c/Users/satar/OneDrive/Desktop/TRACKTER/CODE/COMPUTER_VISION/KEY_FRAME_SELECTION/TRAINING/yolov4-tiny/training/yolov4-tiny-custom_best.weights"
data_path = "/mnt/c/Users/satar/OneDrive/Desktop/TRACKTER/CODE/COMPUTER_VISION/KEY_FRAME_SELECTION/RUNNING/darknet/darknet/data/obj.data"
video_path = "/mnt/c/Users/satar/OneDrive/Desktop/TRACKTER/CODE/COMPUTER_VISION/KEY_FRAME_SELECTION/RUNNING/test.mp4"
output_dir = "/mnt/c/Users/satar/OneDrive/Desktop/TRACKTER/CODE/COMPUTER_VISION/KEY_FRAME_SELECTION/RUNNING/output"


# Tracking parameters
min_detection_confidence = 0.3  # Reduced threshold to 30%
max_age_since_last_detection = 5  # Increased for fault tolerance
min_track_points = 8  # Minimum feature points for tracking
max_distance = 50  # Max pixel distance for matching
reid_threshold = 0.7  # Minimum IOU for re-identification

lk_params = dict(winSize=(15, 15),
                maxLevel=2,
                criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

# ===== Initialization =====
def verify_paths():
    """Check all required files exist and have proper permissions"""
    if not os.path.exists(darknet_path):
        raise FileNotFoundError(f"Darknet binary not found at {darknet_path}")
    
    required_files = {
        "Config": cfg_path,
        "Weights": weights_path,
        "Data": data_path,
        "Video": video_path
    }
    
    for name, path in required_files.items():
        if not os.path.exists(path):
            raise FileNotFoundError(f"{name} file not found at {path}")
    
    if not os.access(darknet_path, os.X_OK):
        print("Making darknet executable...")
        try:
            os.chmod(darknet_path, 0o755)  # rwxr-xr-x
        except PermissionError:
            raise PermissionError(f"Could not make {darknet_path} executable")

verify_paths()
os.makedirs(output_dir, exist_ok=True)

# Initialize video capture
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise IOError(f"Cannot open video file {video_path}")

# Get video properties
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_interval = 2 * fps  # Process every 2 seconds
print(f"Video: {width}x{height} @ {fps}fps")

# Tracking data structure
tracked_objects = {}
next_id = 1

# ===== Core Functions =====
def parse_detection_output(output):
    """Robust parser for YOLO detection output"""
    detections = []
    # Pattern for: "class: confidence% (left_x: X top_y: Y width: W height: H)"
    pattern = r'(\w+):\s*(\d+)%\s*\(left_x:\s*(\d+)\s*top_y:\s*(\d+)\s*width:\s*(\d+)\s*height:\s*(\d+)\)'
    matches = re.findall(pattern, output)
    
    for match in matches:
        try:
            confidence = float(match[1]) / 100
            if confidence >= min_detection_confidence:
                detections.append({
                    'class': match[0],
                    'confidence': confidence,
                    'bbox': [int(match[2]), int(match[3]), int(match[4]), int(match[5])]
                })
        except Exception as e:
            print(f"Error parsing detection: {e}")
    
    print(f"Found {len(detections)} valid detections")
    return detections

def calculate_iou(box1, box2):
    """Calculate Intersection over Union for two bounding boxes"""
    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2
    
    xi1 = max(x1, x2)
    yi1 = max(y1, y2)
    xi2 = min(x1+w1, x2+w2)
    yi2 = min(y1+h1, y2+h2)
    
    inter_area = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    box1_area = w1 * h1
    box2_area = w2 * h2
    
    return inter_area / float(box1_area + box2_area - inter_area)

def match_detections_to_tracks(detections, tracked_objects):
    """Improved matching with IOU for re-identification"""
    matches = {}
    unmatched_detections = list(range(len(detections))) if detections else []
    unmatched_tracks = list(tracked_objects.keys()) if tracked_objects else []
    
    if not detections or not tracked_objects:
        return matches, unmatched_detections, unmatched_tracks
    
    # First try matching with Hungarian algorithm on center distance
    cost_matrix = np.zeros((len(tracked_objects), len(detections)))
    for i, (track_id, obj) in enumerate(tracked_objects.items()):
        for j, det in enumerate(detections):
            track_center = np.array([obj['bbox'][0] + obj['bbox'][2]/2, 
                                    obj['bbox'][1] + obj['bbox'][3]/2])
            det_center = np.array([det['bbox'][0] + det['bbox'][2]/2,
                                  det['bbox'][1] + det['bbox'][3]/2])
            cost_matrix[i,j] = np.linalg.norm(track_center - det_center)
    
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    
    # Verify matches with IOU threshold
    for i, j in zip(row_ind, col_ind):
        track_id = list(tracked_objects.keys())[i]
        det = detections[j]
        iou = calculate_iou(tracked_objects[track_id]['bbox'], det['bbox'])
        
        if iou > reid_threshold or cost_matrix[i,j] < max_distance:
            matches[track_id] = j
    
    # Update unmatched lists
    unmatched_detections = [j for j in range(len(detections)) if j not in matches.values()]
    unmatched_tracks = [track_id for track_id in tracked_objects if track_id not in matches]
    
    return matches, unmatched_detections, unmatched_tracks

# ===== Main Processing Loop =====
prev_frame = None
frame_count = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame_count % frame_interval == 0:
        print(f"\nProcessing frame {frame_count}")
        
        # Save frame for YOLO
        frame_filename = os.path.join(output_dir, f"frame_{frame_count:04d}.jpg")
        cv2.imwrite(frame_filename, frame)
        
        # Run YOLO detection
        cmd = [
            darknet_path,
            'detector', 'test', 
            data_path, cfg_path, weights_path,
            frame_filename,
            '-thresh', str(min_detection_confidence),
            '-ext_output', '-dont_show'
        ]
        result = subprocess.run(cmd, capture_output=True, text=True)
        detections = parse_detection_output(result.stdout)
        
        current_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        if prev_frame is None:
            # First frame initialization
            prev_frame = frame.copy()
            prev_gray = current_gray.copy()
            
            # Initialize tracks only from high-confidence detections
            for det in detections:
                x, y, w, h = det['bbox']
                mask = np.zeros_like(prev_gray)
                mask[y:y+h, x:x+w] = 255
                points = cv2.goodFeaturesToTrack(prev_gray, mask=mask,
                                               maxCorners=100,
                                               qualityLevel=0.3,
                                               minDistance=7)
                if points is not None and len(points) >= min_track_points:
                    tracked_objects[next_id] = {
                        'bbox': det['bbox'],
                        'points': points,
                        'class': det['class'],
                        'age': 1,
                        'detected': True,
                        'history': [det['bbox']]  # Initialize history
                    }
                    next_id += 1
        else:
            # Update existing tracks with optical flow
            for obj_id in list(tracked_objects.keys()):
                old_points = tracked_objects[obj_id]['points']
                new_points, status, _ = cv2.calcOpticalFlowPyrLK(
                    prev_gray, current_gray, old_points, None, **lk_params)
                
                good_new = new_points[status.flatten() == 1]
                
                if len(good_new) >= min_track_points:
                    # Update track
                    tracked_objects[obj_id]['points'] = good_new.reshape(-1, 1, 2)
                    tracked_objects[obj_id]['age'] += 1
                    tracked_objects[obj_id]['detected'] = False
                    
                    # Update bbox position
                    if tracked_objects[obj_id]['bbox']:
                        x, y, w, h = tracked_objects[obj_id]['bbox']
                        median_flow = np.median(good_new - old_points[status.flatten() == 1], axis=0)
                        new_bbox = (int(x + median_flow[0,0]),
                                   int(y + median_flow[0,1]), w, h)
                        tracked_objects[obj_id]['bbox'] = new_bbox
                        tracked_objects[obj_id]['history'].append(new_bbox)
                else:
                    del tracked_objects[obj_id]
            
            # Match detections to tracks
            matches, unmatched_dets, unmatched_tracks = match_detections_to_tracks(detections, tracked_objects)
            
            # Secondary matching for reappearing objects
            for track_id in unmatched_tracks.copy():
                for det_idx in unmatched_dets.copy():
                    iou = calculate_iou(tracked_objects[track_id]['bbox'], detections[det_idx]['bbox'])
                    if iou > reid_threshold:
                        matches[track_id] = det_idx
                        unmatched_tracks.remove(track_id)
                        unmatched_dets.remove(det_idx)
                        break
            
            # Update matched tracks
            for track_id, det_idx in matches.items():
                det = detections[det_idx]
                tracked_objects[track_id].update({
                    'bbox': det['bbox'],
                    'class': det['class'],
                    'detected': True,
                    'age': 1  # Reset age counter
                })
                tracked_objects[track_id]['history'].append(det['bbox'])
                
                # Refresh tracking points
                x, y, w, h = det['bbox']
                mask = np.zeros_like(current_gray)
                mask[y:y+h, x:x+w] = 255
                points = cv2.goodFeaturesToTrack(current_gray, mask=mask,
                                               maxCorners=100,
                                               qualityLevel=0.3,
                                               minDistance=7)
                if points is not None:
                    tracked_objects[track_id]['points'] = points
            
            # Create new tracks for unmatched detections
            for det_idx in unmatched_dets:
                det = detections[det_idx]
                # Check if this might be a reappeared track
                possible_matches = []
                for track_id in unmatched_tracks:
                    iou = calculate_iou(tracked_objects[track_id]['bbox'], det['bbox'])
                    if iou > reid_threshold/2:  # Lower threshold for reappearance
                        possible_matches.append((track_id, iou))
                
                if possible_matches:
                    # Reconnect to best matching old track
                    best_match = max(possible_matches, key=lambda x: x[1])
                    track_id = best_match[0]
                    matches[track_id] = det_idx
                    tracked_objects[track_id].update({
                        'bbox': det['bbox'],
                        'class': det['class'],
                        'detected': True,
                        'age': 1
                    })
                    tracked_objects[track_id]['history'].append(det['bbox'])
                else:
                    # Create new track
                    x, y, w, h = det['bbox']
                    mask = np.zeros_like(current_gray)
                    mask[y:y+h, x:x+w] = 255
                    points = cv2.goodFeaturesToTrack(current_gray, mask=mask,
                                                   maxCorners=100,
                                                   qualityLevel=0.3,
                                                   minDistance=7)
                    if points is not None and len(points) >= min_track_points:
                        tracked_objects[next_id] = {
                            'bbox': det['bbox'],
                            'points': points,
                            'class': det['class'],
                            'age': 1,
                            'detected': True,
                            'history': [det['bbox']]
                        }
                        next_id += 1
            
            # Remove old undetected tracks
            for track_id in unmatched_tracks:
                if tracked_objects[track_id]['age'] > max_age_since_last_detection:
                    del tracked_objects[track_id]
        
        # Visualization
        output_frame = frame.copy()
        for obj_id, obj in tracked_objects.items():
            if not obj['bbox']:
                continue
                
            x, y, w, h = obj['bbox']
            if obj['detected']:
                color = (0, 255, 0)  # Green for detected
                thickness = 2
            else:
                color = (0, 0, 255)  # Red for predicted
                thickness = 1
                
            cv2.rectangle(output_frame, (x, y), (x+w, y+h), color, thickness)
            label = f"{obj_id}:{obj['class']}"
            cv2.putText(output_frame, label, (x, y-10), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
        
        # Save output
        output_filename = os.path.join(output_dir, f"frame_{frame_count:04d}_tracked.jpg")
        cv2.imwrite(output_filename, output_frame)
        
        prev_frame = frame.copy()
        prev_gray = current_gray.copy()

    frame_count += 1

cap.release()
print(f"\nProcessing complete. Final track count: {next_id-1}")

Video: 1920x1020 @ 29fps

Processing frame 0
Found 2 valid detections

Processing frame 58
Found 2 valid detections

Processing frame 116
Found 2 valid detections

Processing frame 174
Found 3 valid detections

Processing frame 232
Found 2 valid detections

Processing frame 290
Found 2 valid detections

Processing frame 348
Found 2 valid detections

Processing frame 406
Found 2 valid detections

Processing frame 464
Found 3 valid detections

Processing frame 522
Found 2 valid detections

Processing frame 580
Found 2 valid detections

Processing frame 638
Found 2 valid detections

Processing frame 696
Found 2 valid detections

Processing frame 754
Found 2 valid detections

Processing frame 812
Found 2 valid detections

Processing frame 870
Found 2 valid detections

Processing frame 928


KeyboardInterrupt: 